In [15]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [16]:
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_networkx
from torch_geometric.nn.conv import GINEConv
from torch_geometric.nn.aggr import MeanAggregation, SumAggregation

from tqdm.auto import tqdm

from bcos.modules import BcosLinear
from bcosgnn.explain_edge_attr import explain as explain_edge_attr
import os
import shutil
from torch_geometric.data import InMemoryDataset
import tqdm
import sys
import os
# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

import functools
import itertools
import operator
from typing import Any
import torch
from torch_geometric.data import Dataset, download_url
from torch.utils.data import random_split
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from bcos.modules import BcosLinear, BcosSequential
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation
from torch_geometric.utils import add_self_loops, degree
from torchmetrics import AUROC
from torchmetrics.classification import BinaryAccuracy
from tqdm import tqdm
import networkx as nx
import torch.nn.functional as F
from bcosgnn.explain import explain
from bcosgnn.evaluation import get_attribution_scores

In [17]:
import importlib
import bcosgnn.evaluation
importlib.reload(bcosgnn.evaluation)
from bcosgnn.evaluation import get_attribution_scores

In [18]:
import torch
import torch.nn.functional as F
from torch.nn import Sequential, Linear, BatchNorm1d, ReLU
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool

In [19]:
dataset = TUDataset(root='data/TUDataset', name='MUTAG')

In [20]:
torch.manual_seed(12345)
dataset = dataset.shuffle()
train_dataset = dataset[:150]
test_dataset = dataset[150:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [21]:
class GINEVirtualNode(torch.nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim, num_classes, num_layers=3):
        super(GINEVirtualNode, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Project raw node/edge features to the same hidden dimension
        # (Required for GINEConv, which adds edge features to node features)
        self.node_encoder = Linear(node_in_dim, hidden_dim)
        self.edge_encoder = Linear(edge_in_dim, hidden_dim)

        # The initial embedding for the virtual nodes
        self.virtualnode_embedding = torch.nn.Embedding(1, hidden_dim)

        self.convs = torch.nn.ModuleList()
        self.vn_mlps = torch.nn.ModuleList()

        for _ in range(num_layers):
            # GINE Convolution MLP
            mlp = Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(nn=mlp, train_eps=True))
            
            # Virtual Node Update MLP (one for each layer except the last)
            vn_mlp = Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim)
            )
            self.vn_mlps.append(vn_mlp)

        # Final classifier
        self.lin = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr, batch):
        # 1. Encode initial features
        x = self.node_encoder(x.float())
        edge_attr = self.edge_encoder(edge_attr.float())

        # 2. Initialize Virtual Node for this batch
        # We need one virtual node per graph in the batch.
        # batch.max().item() + 1 gives the number of graphs in the current batch.
        device = x.device
        num_graphs = batch.max().item() + 1
        
        # Start with the learned initial embedding
        vn_state = self.virtualnode_embedding(torch.zeros(num_graphs, dtype=torch.long, device=device))

        # 3. Message Passing Loop
        for i in range(self.num_layers):
            # A. Add Virtual Node information to physical nodes
            # vn_state[batch] broadcasts the graph-level vn to all its respective nodes
            x = x + vn_state[batch]

            # B. Standard GINE Convolution
            x = self.convs[i](x, edge_index, edge_attr)
            x = F.relu(x)

            # C. Update the Virtual Node (if not the last layer)
            if i < self.num_layers - 1:
                # Pool physical nodes to create a global summary
                global_summary = global_add_pool(x, batch)
                # Combine old VN state with new global summary and pass through MLP
                vn_state = self.vn_mlps[i](vn_state + global_summary)
                vn_state = F.relu(vn_state)

        # 4. Final Graph Readout
        # Pool the final node embeddings to get a graph-level representation
        out = global_mean_pool(x, batch) 
        
        # 5. Classify
        out = F.dropout(out, p=0.5, training=self.training)
        out = self.lin(out)
        
        return out

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [23]:
model = GINEVirtualNode(node_in_dim=7, edge_in_dim=4, hidden_dim=32, num_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

In [24]:
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_loader.dataset)

In [25]:
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return correct / len(loader.dataset)

In [26]:
print(f"Running on {device}...")
for epoch in range(1, 51):
    loss = train()
    train_acc = test(train_loader)
    test_acc = test(test_loader)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

Running on cpu...
Epoch: 001, Loss: 0.6124, Train Acc: 0.6467, Test Acc: 0.7368
Epoch: 010, Loss: 0.2937, Train Acc: 0.7467, Test Acc: 0.8158
Epoch: 010, Loss: 0.2937, Train Acc: 0.7467, Test Acc: 0.8158
Epoch: 020, Loss: 0.1904, Train Acc: 0.8667, Test Acc: 0.8421
Epoch: 020, Loss: 0.1904, Train Acc: 0.8667, Test Acc: 0.8421
Epoch: 030, Loss: 0.2450, Train Acc: 0.8200, Test Acc: 0.7895
Epoch: 030, Loss: 0.2450, Train Acc: 0.8200, Test Acc: 0.7895
Epoch: 040, Loss: 0.1553, Train Acc: 0.8067, Test Acc: 0.8421
Epoch: 040, Loss: 0.1553, Train Acc: 0.8067, Test Acc: 0.8421
Epoch: 050, Loss: 0.1661, Train Acc: 0.8133, Test Acc: 0.8684
Epoch: 050, Loss: 0.1661, Train Acc: 0.8133, Test Acc: 0.8684


In [27]:
import torch
import torch.nn.functional as F
from torch.nn import Sequential, Linear, BatchNorm1d, ReLU
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from sklearn.metrics import f1_score

# ==========================================
# 1. Dataset Loading and Strict Splitting
# ==========================================
# MUTAG has 188 total graphs (binary classification: mutagenic vs non-mutagenic).
# We split it: ~80% Train (150), ~10% Val (18), ~10% Test (20)
dataset = TUDataset(root='data/TUDataset', name='MUTAG')

torch.manual_seed(12345)
dataset = dataset.shuffle()

train_dataset = dataset[:150]
val_dataset = dataset[150:168]
test_dataset = dataset[168:]

# DataLoaders for batching
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 2. GINE + Virtual Node Architecture
# ==========================================
class GINEVirtualNode(torch.nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim, num_classes, num_layers=3):
        super(GINEVirtualNode, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Project raw node/edge features to the same hidden dimension
        self.node_encoder = Linear(node_in_dim, hidden_dim)
        self.edge_encoder = Linear(edge_in_dim, hidden_dim)

        # Initial embedding for the virtual nodes
        self.virtualnode_embedding = torch.nn.Embedding(1, hidden_dim)

        self.convs = torch.nn.ModuleList()
        self.vn_mlps = torch.nn.ModuleList()

        for _ in range(num_layers):
            # GINE Convolution MLP
            mlp = Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(nn=mlp, train_eps=True))
            
            # Virtual Node Update MLP
            vn_mlp = Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim)
            )
            self.vn_mlps.append(vn_mlp)

        # Final graph-level classifier
        self.lin = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr, batch):
        # Encode initial features
        x = self.node_encoder(x.float())
        edge_attr = self.edge_encoder(edge_attr.float())

        device = x.device
        # Determine number of graphs in the current batch to initialize VNs
        num_graphs = batch.max().item() + 1
        
        # Start with the learned initial embedding for the VN state
        vn_state = self.virtualnode_embedding(torch.zeros(num_graphs, dtype=torch.long, device=device))

        # Message Passing Loop
        for i in range(self.num_layers):
            # 1. Add Virtual Node information to physical nodes
            x = x + vn_state[batch]

            # 2. Standard GINE Convolution
            x = self.convs[i](x, edge_index, edge_attr)
            x = F.relu(x)

            # 3. Update the Virtual Node state (if not the last layer)
            if i < self.num_layers - 1:
                global_summary = global_add_pool(x, batch)
                vn_state = self.vn_mlps[i](vn_state + global_summary)
                vn_state = F.relu(vn_state)

        # Final Graph Readout (Mean pooling across physical nodes)
        out = global_mean_pool(x, batch) 
        
        # Classification
        out = F.dropout(out, p=0.5, training=self.training)
        out = self.lin(out)
        
        return out

# ==========================================
# 3. Setup, Train, and Evaluate Functions
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# MUTAG features: 7 node features, 4 edge features, 2 classes
model = GINEVirtualNode(node_in_dim=7, edge_in_dim=4, hidden_dim=64, num_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_loader.dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    correct = 0
    
    all_preds = []
    all_labels = []
    
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        
        # Loss calculation
        loss = criterion(out, data.y)
        total_loss += loss.item() * data.num_graphs
        
        # Accuracy calculation
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
        
        # Collect for F1 Score
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(data.y.cpu().numpy())
        
    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, accuracy, f1

# ==========================================
# 4. Main Execution Loop
# ==========================================
if __name__ == "__main__":
    print(f"Running on {device}...\n")
    print("--- Starting Training ---")
    
    # Train and Validate
    for epoch in range(1, 101):
        train_loss = train()
        _, train_acc, train_f1 = evaluate(train_loader) 
        val_loss, val_acc, val_f1 = evaluate(val_loader)
        
        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch: {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} || Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

    # Final Inference isolated entirely to the Test Set
    print("\n--- Final Test Inference ---")
    test_loss, test_acc, test_f1 = evaluate(test_loader)
    print(f'Test Loss:       {test_loss:.4f}')
    print(f'Test Accuracy:   {test_acc:.4f}')
    print(f'Test F1 (Macro): {test_f1:.4f}')

Running on cpu...

--- Starting Training ---
Epoch: 001 | Train Loss: 0.5799 | Val Loss: 85.0962 || Val Acc: 0.7222 | Val F1: 0.4194
Epoch: 010 | Train Loss: 0.3386 | Val Loss: 0.6422 || Val Acc: 0.7778 | Val F1: 0.6000
Epoch: 010 | Train Loss: 0.3386 | Val Loss: 0.6422 || Val Acc: 0.7778 | Val F1: 0.6000
Epoch: 020 | Train Loss: 0.2619 | Val Loss: 0.7442 || Val Acc: 0.8333 | Val F1: 0.7340
Epoch: 020 | Train Loss: 0.2619 | Val Loss: 0.7442 || Val Acc: 0.8333 | Val F1: 0.7340
Epoch: 030 | Train Loss: 0.3287 | Val Loss: 0.9974 || Val Acc: 0.7222 | Val F1: 0.4194
Epoch: 030 | Train Loss: 0.3287 | Val Loss: 0.9974 || Val Acc: 0.7222 | Val F1: 0.4194
Epoch: 040 | Train Loss: 0.2341 | Val Loss: 0.2381 || Val Acc: 0.8333 | Val F1: 0.8194
Epoch: 040 | Train Loss: 0.2341 | Val Loss: 0.2381 || Val Acc: 0.8333 | Val F1: 0.8194
Epoch: 050 | Train Loss: 0.2126 | Val Loss: 0.4484 || Val Acc: 0.7778 | Val F1: 0.7662
Epoch: 050 | Train Loss: 0.2126 | Val Loss: 0.4484 || Val Acc: 0.7778 | Val F1: 0.76

## Vanilla GINE + Virtual Node

### Model Definitions (Vanilla Only)

In [28]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_curve
from torch_geometric.nn import GINEConv, global_add_pool

class VanillaGINE(torch.nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=3, num_classes=2, dropout=0.5):
        super().__init__()
        self.lin_node = nn.Linear(node_dim, hidden_dim)
        self.lin_edge = nn.Linear(edge_dim, hidden_dim)

        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(nn=mlp, train_eps=True))

        self.post_conv = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        edge_attr = self.lin_edge(edge_attr)

        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
            x = torch.relu(x)

        x = global_add_pool(x, batch)
        return self.post_conv(x)


class VanillaGINEVirtualNode(torch.nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, num_layers=3, num_classes=2, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.edge_encoder = nn.Linear(edge_dim, hidden_dim)
        self.virtualnode_embedding = nn.Embedding(1, hidden_dim)

        self.convs = nn.ModuleList()
        self.vn_mlps = nn.ModuleList()

        for _ in range(num_layers):
            conv_mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(nn=conv_mlp, train_eps=True))

            vn_mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.vn_mlps.append(vn_mlp)

        self.post_conv = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.node_encoder(x.float())
        edge_attr = self.edge_encoder(edge_attr.float())

        num_graphs = int(batch.max().item()) + 1
        vn_state = self.virtualnode_embedding(
            torch.zeros(num_graphs, dtype=torch.long, device=x.device)
        )

        for layer_idx in range(self.num_layers):
            x = x + vn_state[batch]
            x = self.convs[layer_idx](x, edge_index, edge_attr)
            x = F.relu(x)

            if layer_idx < self.num_layers - 1:
                summary = global_add_pool(x, batch)
                vn_state = self.vn_mlps[layer_idx](vn_state + summary)
                vn_state = F.relu(vn_state)

        x = global_add_pool(x, batch)
        return self.post_conv(x)

In [31]:
import copy
import torch.optim as optim
from sklearn.metrics import f1_score
from torch_geometric.loader import DataLoader

def compute_optimal_f1(y_true, y_logits):
    y_probs = torch.sigmoid(torch.tensor(y_logits)).numpy()
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]

    if len(f1_scores) == 0:
        return 0.0, 0.0, 0.5

    best_idx = int(np.argmax(f1_scores))
    best_f1 = float(f1_scores[best_idx])
    best_thresh = float(thresholds[best_idx])
    y_pred_optimal = (y_probs >= best_thresh).astype(int)
    best_acc = float(accuracy_score(y_true, y_pred_optimal))
    return best_f1, best_acc, best_thresh


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def collect_scores(model, loader, device):
    model.eval()
    y_true, y_logits = [], []
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        score = out[:, 1] - out[:, 0]
        y_true.extend(data.y.cpu().numpy())
        y_logits.extend(score.cpu().numpy())
    return np.array(y_true), np.array(y_logits)


@torch.no_grad()
def evaluate_argmax(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        pred = out.argmax(dim=1)
        y_true.extend(data.y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
    f1 = f1_score(y_true, y_pred, average="binary")
    acc = accuracy_score(y_true, y_pred)
    return float(f1), float(acc)


def evaluate_with_threshold(model, loader, device, threshold):
    y_true, y_logits = collect_scores(model, loader, device)
    y_probs = torch.sigmoid(torch.tensor(y_logits)).numpy()
    y_pred = (y_probs >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred, average="binary")
    acc = accuracy_score(y_true, y_pred)
    return float(f1), float(acc)


def run_seed_experiment(model_cls, model_name, dataset, train_dataset, val_dataset, test_dataset):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seeds = [42, 123, 999, 7, 2024]

    HIDDEN_DIM = 64
    NUM_LAYERS = 3
    DROPOUT = 0.5
    LR = 0.001
    EPOCHS = 100
    BATCH_SIZE = 64

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    edge_dim = dataset[0].edge_attr.size(1) if dataset[0].edge_attr is not None else 1
    results = {
        "val_f1": [],
        "test_f1_dynamic": [],
        "test_acc_dynamic": [],
        "test_f1_fixed": [],
        "test_acc_fixed": [],
        "val_thresh": [],
    }

    print(f"\n--- Running {model_name} ---")
    print("Dynamic = val-selected threshold | Fixed = argmax (no threshold tuning)")

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = model_cls(
            node_dim=dataset.num_features,
            edge_dim=edge_dim,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            num_classes=dataset.num_classes,
            dropout=DROPOUT,
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()

        best_val_f1 = -1.0
        best_val_thresh = 0.5
        best_state = copy.deepcopy(model.state_dict())

        for _ in range(EPOCHS):
            _ = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_true, val_logits = collect_scores(model, val_loader, device)
            val_f1, _, val_thresh = compute_optimal_f1(val_true, val_logits)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_val_thresh = val_thresh
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)

        test_f1_dynamic, test_acc_dynamic = evaluate_with_threshold(
            model, test_loader, device, best_val_thresh
        )
        test_f1_fixed, test_acc_fixed = evaluate_argmax(model, test_loader, device)

        results["val_f1"].append(best_val_f1)
        results["test_f1_dynamic"].append(test_f1_dynamic)
        results["test_acc_dynamic"].append(test_acc_dynamic)
        results["test_f1_fixed"].append(test_f1_fixed)
        results["test_acc_fixed"].append(test_acc_fixed)
        results["val_thresh"].append(best_val_thresh)

        print(
            f"Seed {seed}: Val F1={best_val_f1:.4f} | "
            f"Dynamic Test F1={test_f1_dynamic:.4f}, Acc={test_acc_dynamic:.4f} | "
            f"Fixed Test F1={test_f1_fixed:.4f}, Acc={test_acc_fixed:.4f} | "
            f"Val Thresh={best_val_thresh:.3f}"
        )

    print(f"\n=== {model_name} Final Results (5 Seeds) ===")
    print(
        f"Dynamic Test F1: {np.mean(results['test_f1_dynamic']):.4f} "
        f"± {np.std(results['test_f1_dynamic']):.4f}"
    )
    print(
        f"Dynamic Test Acc: {np.mean(results['test_acc_dynamic']):.4f} "
        f"± {np.std(results['test_acc_dynamic']):.4f}"
    )
    print(
        f"Fixed Test F1: {np.mean(results['test_f1_fixed']):.4f} "
        f"± {np.std(results['test_f1_fixed']):.4f}"
    )
    print(
        f"Fixed Test Acc: {np.mean(results['test_acc_fixed']):.4f} "
        f"± {np.std(results['test_acc_fixed']):.4f}"
    )

    return results


vanilla_results = run_seed_experiment(
    VanillaGINE,
    "Vanilla GINE",
    dataset,
    train_dataset,
    val_dataset,
    test_dataset,
 )

vanilla_vn_results = run_seed_experiment(
    VanillaGINEVirtualNode,
    "Vanilla GINE + Virtual Node",
    dataset,
    train_dataset,
    val_dataset,
    test_dataset,
 )

print("\n=== Summary Comparison (Test Set) ===")
print(
    f"Vanilla GINE      | Dynamic F1: {np.mean(vanilla_results['test_f1_dynamic']):.4f} | "
    f"Fixed F1: {np.mean(vanilla_results['test_f1_fixed']):.4f}"
)
print(
    f"Vanilla GINE + VN | Dynamic F1: {np.mean(vanilla_vn_results['test_f1_dynamic']):.4f} | "
    f"Fixed F1: {np.mean(vanilla_vn_results['test_f1_fixed']):.4f}"
)


--- Running Vanilla GINE ---
Dynamic = val-selected threshold | Fixed = argmax (no threshold tuning)
Seed 42: Val F1=0.9231 | Dynamic Test F1=0.9032, Acc=0.8500 | Fixed Test F1=0.8571, Acc=0.7500 | Val Thresh=0.629
Seed 42: Val F1=0.9231 | Dynamic Test F1=0.9032, Acc=0.8500 | Fixed Test F1=0.8571, Acc=0.7500 | Val Thresh=0.629
Seed 123: Val F1=0.9231 | Dynamic Test F1=0.8667, Acc=0.8000 | Fixed Test F1=0.8966, Acc=0.8500 | Val Thresh=0.302
Seed 123: Val F1=0.9231 | Dynamic Test F1=0.8667, Acc=0.8000 | Fixed Test F1=0.8966, Acc=0.8500 | Val Thresh=0.302
Seed 999: Val F1=0.9286 | Dynamic Test F1=0.9032, Acc=0.8500 | Fixed Test F1=0.8966, Acc=0.8500 | Val Thresh=0.284
Seed 999: Val F1=0.9286 | Dynamic Test F1=0.9032, Acc=0.8500 | Fixed Test F1=0.8966, Acc=0.8500 | Val Thresh=0.284
Seed 7: Val F1=0.9231 | Dynamic Test F1=0.8966, Acc=0.8500 | Fixed Test F1=0.8571, Acc=0.8000 | Val Thresh=0.262
Seed 7: Val F1=0.9231 | Dynamic Test F1=0.8966, Acc=0.8500 | Fixed Test F1=0.8571, Acc=0.8000 | V

# B-Cos GINE

## Bcos Model Defintion

In [32]:
class Readout(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels=None,
        out_channels=1,
        b=2,
        max_out=1,
        agg: str = "sum",
    ):
        super().__init__()
        if hidden_channels is None:
            self.readout = BcosLinear(in_channels, out_channels, b=b, max_out=max_out)
        else:
            hidden_channels = (
                [hidden_channels]
                if isinstance(hidden_channels, int)
                else hidden_channels
            )
            channels = [in_channels] + hidden_channels + [out_channels]
            self.readout = BcosSequential(
                *[
                    BcosLinear(d_in, d_out, b=b, max_out=max_out)
                    for d_in, d_out in zip(channels[:-1], channels[1:])
                ]
            )
        match agg:
            case "sum":
                self.agg = SumAggregation()
            case _:
                raise ValueError(f"Aggregation '{agg}' not supported.")

    def forward(self, x, batch):
        raise NotImplementedError


class AggThenReadout(Readout):
    def forward(self, x, batch):
        z = self.agg(x, batch)
        out = self.readout(z)
        return out

class ReadoutThenAgg(Readout):
    def forward(self, x, batch):
        z = self.readout(x)
        out = self.agg(z, batch)
        return out

import torch
from torch_geometric.nn import MessagePassing

class BcosGINEConv(MessagePassing):
    def __init__(
        self,
        channels: list[int],
        edge_dim: int,
        b: float = 2.0,
        max_out: int = 1,
        eps: float = 0.0,
        train_eps: bool = False,
        **kwargs
    ):
        # We use 'add' aggregation to stay true to the GIN formula
        kwargs.setdefault("aggr", "add")
        super().__init__(**kwargs)
        
        # The MLP part of GIN, but using B-cos layers
        self.transform = BcosSequential(
            *[
                BcosLinear(din, dout, b=b, max_out=max_out)
                for din, dout in zip(channels[:-1], channels[1:])
            ]
        )
        
        self.initial_eps = eps
        if train_eps:
            self.eps = torch.nn.Parameter(torch.Tensor([eps]))
        else:
            self.register_buffer("eps", torch.Tensor([eps]))

    def forward(self, x, edge_index, edge_attr):
        # edge_attr is expected to be pre-projected to match x dimension
        # in your main model loop.
        
        # 1. Propagate messages
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        
        # 2. Combine step: (1 + eps) * center_node + aggregated_messages
        out = (1 + self.eps) * x + out
        
        # 3. Apply the B-cos MLP transformation
        return self.transform(out)

    def message(self, x_j, edge_attr):
        # Standard GINE uses ReLU(x_j + edge_attr).
        # In pure B-cos, we can use the addition, then the B-cos transform 
        # in the 'forward' call handles the non-linear alignment.
        return torch.nn.functional.relu(x_j + edge_attr)

In [34]:
class PureBcosGINE(nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 4,
        num_classes: int = 9,
        b: float = 2.0,
        max_out: int = 1,
        dropout: float = 0.5,
    ):
        super().__init__()

        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)

        self.convs = nn.ModuleList([
            BcosGINEConv(
                channels=[hidden_dim, hidden_dim],
                edge_dim=hidden_dim,
                b=b,
                max_out=max_out,
            )
            for _ in range(num_layers)
        ])

        self.readout_mlp = BcosSequential(
            BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out),
        )

        self.dropout_layer = nn.Dropout(dropout)
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        e = self.lin_edge(edge_attr)

        for conv in self.convs:
            x = conv(x, edge_index, e)

        node_logits = self.readout_mlp(x)
        node_logits = self.dropout_layer(node_logits)
        graph_logits = self.agg(node_logits, batch)
        return graph_logits


class PureBcosGINEVirtualNode(nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 4,
        num_classes: int = 9,
        b: float = 2.0,
        max_out: int = 1,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.num_layers = num_layers

        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)
        self.virtualnode_embedding = nn.Embedding(1, hidden_dim)

        self.convs = nn.ModuleList([
            BcosGINEConv(
                channels=[hidden_dim, hidden_dim],
                edge_dim=hidden_dim,
                b=b,
                max_out=max_out,
            )
            for _ in range(num_layers)
        ])

        self.vn_mlps = nn.ModuleList([
            BcosSequential(
                BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
                BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            )
            for _ in range(max(1, num_layers - 1))
        ])

        self.readout_mlp = BcosSequential(
            BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out),
        )

        self.dropout_layer = nn.Dropout(dropout)
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)
        e = self.lin_edge(edge_attr)

        num_graphs = int(batch.max().item()) + 1
        vn_state = self.virtualnode_embedding(
            torch.zeros(num_graphs, dtype=torch.long, device=x.device)
        )

        for layer_idx, conv in enumerate(self.convs):
            x = x + vn_state[batch]
            x = conv(x, edge_index, e)
            x = F.relu(x)

            if layer_idx < self.num_layers - 1:
                summary = self.agg(x, batch)
                vn_state = self.vn_mlps[layer_idx](vn_state + summary)
                vn_state = F.relu(vn_state)

        node_logits = self.readout_mlp(x)
        node_logits = self.dropout_layer(node_logits)
        graph_logits = self.agg(node_logits, batch)
        return graph_logits

In [36]:
import copy
import torch.optim as optim

def run_bcos_seed_experiment(model_cls, model_name, dataset, train_dataset, val_dataset, test_dataset):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seeds = [42, 123, 999, 7, 2024]

    HIDDEN_DIM = 64
    NUM_LAYERS = 3
    DROPOUT = 0.5
    LR = 0.001
    EPOCHS = 100
    BATCH_SIZE = 64

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    edge_dim = dataset[0].edge_attr.size(1) if dataset[0].edge_attr is not None else 1

    results = {
        "val_f1": [],
        "test_f1_dynamic": [],
        "test_acc_dynamic": [],
        "test_f1_fixed": [],
        "test_acc_fixed": [],
        "val_thresh": [],
    }

    print(f"\n--- Running {model_name} ---")
    print("Dynamic = val-selected threshold | Fixed = argmax (no threshold tuning)")

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = model_cls(
            node_dim=dataset.num_features,
            edge_dim=edge_dim,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            num_classes=dataset.num_classes,
            b=2,
            max_out=1,
            dropout=DROPOUT,
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()

        best_val_f1 = -1.0
        best_val_thresh = 0.5
        best_state = copy.deepcopy(model.state_dict())

        for _ in range(EPOCHS):
            _ = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_true, val_logits = collect_scores(model, val_loader, device)
            val_f1, _, val_thresh = compute_optimal_f1(val_true, val_logits)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_val_thresh = val_thresh
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)

        test_f1_dynamic, test_acc_dynamic = evaluate_with_threshold(
            model, test_loader, device, best_val_thresh
        )
        test_f1_fixed, test_acc_fixed = evaluate_argmax(model, test_loader, device)

        results["val_f1"].append(best_val_f1)
        results["test_f1_dynamic"].append(test_f1_dynamic)
        results["test_acc_dynamic"].append(test_acc_dynamic)
        results["test_f1_fixed"].append(test_f1_fixed)
        results["test_acc_fixed"].append(test_acc_fixed)
        results["val_thresh"].append(best_val_thresh)

        print(
            f"Seed {seed}: Val F1={best_val_f1:.4f} | "
            f"Dynamic Test F1={test_f1_dynamic:.4f}, Acc={test_acc_dynamic:.4f} | "
            f"Fixed Test F1={test_f1_fixed:.4f}, Acc={test_acc_fixed:.4f} | "
            f"Val Thresh={best_val_thresh:.3f}"
        )

    print(f"\n=== {model_name} Final Results (5 Seeds) ===")
    print(
        f"Dynamic Test F1: {np.mean(results['test_f1_dynamic']):.4f} "
        f"± {np.std(results['test_f1_dynamic']):.4f}"
    )
    print(
        f"Dynamic Test Acc: {np.mean(results['test_acc_dynamic']):.4f} "
        f"± {np.std(results['test_acc_dynamic']):.4f}"
    )
    print(
        f"Fixed Test F1: {np.mean(results['test_f1_fixed']):.4f} "
        f"± {np.std(results['test_f1_fixed']):.4f}"
    )
    print(
        f"Fixed Test Acc: {np.mean(results['test_acc_fixed']):.4f} "
        f"± {np.std(results['test_acc_fixed']):.4f}"
    )

    return results


bcos_results = run_bcos_seed_experiment(
    PureBcosGINE,
    "B-Cos GINE",
    dataset,
    train_dataset,
    val_dataset,
    test_dataset,
 )

bcos_vn_results = run_bcos_seed_experiment(
    PureBcosGINEVirtualNode,
    "B-Cos GINE + Virtual Node",
    dataset,
    train_dataset,
    val_dataset,
    test_dataset,
 )

def summarize_model(model_name, results):
    dyn_f1_mean = np.mean(results["test_f1_dynamic"] )
    dyn_f1_std = np.std(results["test_f1_dynamic"] )
    dyn_acc_mean = np.mean(results["test_acc_dynamic"] )
    dyn_acc_std = np.std(results["test_acc_dynamic"] )
    fix_f1_mean = np.mean(results["test_f1_fixed"] )
    fix_f1_std = np.std(results["test_f1_fixed"] )
    fix_acc_mean = np.mean(results["test_acc_fixed"] )
    fix_acc_std = np.std(results["test_acc_fixed"] )
    print(
        f"{model_name:18} | "
        f"Dyn F1: {dyn_f1_mean:.4f}±{dyn_f1_std:.4f}, Dyn Acc: {dyn_acc_mean:.4f}±{dyn_acc_std:.4f} | "
        f"Fix F1: {fix_f1_mean:.4f}±{fix_f1_std:.4f}, Fix Acc: {fix_acc_mean:.4f}±{fix_acc_std:.4f}"
    )

print("\n=== Cross-Model Summary (Test Set, Mean±Std) ===")
summarize_model("Vanilla GINE", vanilla_results)
summarize_model("Vanilla GINE + VN", vanilla_vn_results)
summarize_model("B-Cos GINE", bcos_results)
summarize_model("B-Cos GINE + VN", bcos_vn_results)


--- Running B-Cos GINE ---
Dynamic = val-selected threshold | Fixed = argmax (no threshold tuning)
Seed 42: Val F1=0.9630 | Dynamic Test F1=0.8485, Acc=0.7500 | Fixed Test F1=0.8571, Acc=0.8000 | Val Thresh=0.410
Seed 42: Val F1=0.9630 | Dynamic Test F1=0.8485, Acc=0.7500 | Fixed Test F1=0.8571, Acc=0.8000 | Val Thresh=0.410
Seed 123: Val F1=0.9286 | Dynamic Test F1=0.8276, Acc=0.7500 | Fixed Test F1=0.8889, Acc=0.8500 | Val Thresh=0.194
Seed 123: Val F1=0.9286 | Dynamic Test F1=0.8276, Acc=0.7500 | Fixed Test F1=0.8889, Acc=0.8500 | Val Thresh=0.194
Seed 999: Val F1=0.9286 | Dynamic Test F1=0.7857, Acc=0.7000 | Fixed Test F1=0.5000, Acc=0.5000 | Val Thresh=0.127
Seed 999: Val F1=0.9286 | Dynamic Test F1=0.7857, Acc=0.7000 | Fixed Test F1=0.5000, Acc=0.5000 | Val Thresh=0.127
Seed 7: Val F1=0.9630 | Dynamic Test F1=0.8000, Acc=0.7000 | Fixed Test F1=0.8462, Acc=0.8000 | Val Thresh=0.235
Seed 7: Val F1=0.9630 | Dynamic Test F1=0.8000, Acc=0.7000 | Fixed Test F1=0.8462, Acc=0.8000 | Val

## Virtual Node Flow (Quick Diagram)

Below is one forward-pass view for both **Vanilla+VN** and **B-Cos+VN** (same VN wiring idea).

```text
For each layer l:
  Nodes x^(l), Graph VN state v^(l)

  1) VN -> all nodes (broadcast by batch index)
       x_tilde = x^(l) + v^(l)[batch]

  2) Node update by graph conv
       x_next = Conv(x_tilde, edge_index, edge_attr)

  3) Nodes -> VN (global pooling + VN-MLP)
       s = Pool(x_next, batch)
       v^(l+1) = MLP(v^(l) + s)

Repeat for all layers
```

### Why this means “VN is connected to all nodes”

- `v[batch]` duplicates each graph's VN vector to **every node in that graph**.
- So each node receives graph-global context every layer.
- `Pool(..., batch)` then aggregates **all node info** back into one graph summary.
- This creates a full two-way global channel at each layer:
  - **VN -> nodes** (broadcast add)
  - **nodes -> VN** (pool + update)

No explicit VN edges are required in `edge_index`; the tensor broadcast + pooling performs the same role.